# SAC Navigation — segway_1d_wheel

**Ozzy physics, 1-D only (reaction wheel removed)**

| | Ozzy (segway_2) | segway_1d_wheel |
|---|---|---|
| Ground wheel | real contact + friction=2 | same |
| Reaction wheel | yes | **removed** |
| Integrator | default | **Euler** |
| ctrl sign | -u | same |
| State | qpos[0]=x, quat→θ, qvel[4]=θ̇ | same |

**Fixes:** `TARGET_ENT=0.0` (no alpha collapse) · no scipy · no ground-z table · print every 10 ep


In [1]:
import mujoco, numpy as np, torch, torch.nn as nn, torch.optim as optim
import os, imageio, time, json
from IPython.display import HTML
from PIL import Image as PILImage, ImageDraw
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

XML_PATH = "segway_1d_wheel.xml"  # <-- update this

SAVE_DIR     = "SavedSeeds"
TRAIN_SEEDS  = [42]
VERIFY_SEEDS = [788, 999, 555, 321, 444]
os.makedirs(SAVE_DIR, exist_ok=True)

TORQUE_MAX   =  5.0    # wheel motor ctrlrange [-5, 5] Nm
GROUND_Z     =  0.075  # wheel radius — hardcoded, no lookup table needed
X_NORM       =  2.4
XD_NORM      =  5.0
TH_NORM      =  1.57
THD_NORM     =  5.0
X_DONE_LIMIT =  2.8
TH_DONE      =  0.5
UPDATE_EVERY    = 4
UPDATE_PER_STEP = 2


## Inline RunLogger


In [3]:
class RunLogger:
    def __init__(self, algo, seed, log_every_steps=2000):
        self.algo=algo; self.seed=seed; self.log_every_steps=log_every_steps
        self.rows=[]; self._ep_rewards=[]; self._ep_arrives=[]; self._ep_losses=[]
        self._next_log=log_every_steps; self._t0=time.time()
    def episode_end(self, total_steps, ep_count, ep_ret, arrived, loss):
        self._ep_rewards.append(ep_ret); self._ep_arrives.append(float(arrived))
        self._ep_losses.append(loss)
        avg=float(np.mean(self._ep_rewards[-100:])); rp=float(np.mean(self._ep_arrives[-100:]))*100.
        if total_steps>=self._next_log:
            self.rows.append({"steps":total_steps,"avg_reward":avg,
                              "loss":float(np.mean(self._ep_losses[-100:])),"wall_time":time.time()-self._t0})
            self._next_log+=self.log_every_steps
        return avg, rp
    def save(self, save_dir, tag, eval_results=None):
        path=f"{save_dir}/{self.algo}_{tag}.json"
        with open(path,"w") as f:
            json.dump({"algo":self.algo,"seed":self.seed,"rows":self.rows,
                       "eval":eval_results,"total_wall_time":self.total_wall_time},f,indent=2)
        print(f"Log saved: {path}")
    @property
    def total_wall_time(self): return time.time()-self._t0


## Observation helpers

Fast inline quaternion formula — no scipy.


In [2]:
def get_obs(data):
    """
    freejoint + wheel hinge:
      qpos = [x, y, z, qw, qx, qy, qz, wheel_angle]
      qvel = [vx, vy, vz, wx, wy, wz, wheel_speed]
    pitch (XYZ Tait-Bryan): arcsin(2*(qw*qy - qz*qx))
    """
    qw,qx,qy,qz = data.qpos[3],data.qpos[4],data.qpos[5],data.qpos[6]
    theta = float(np.arcsin(np.clip(2.0*(qw*qy - qz*qx), -1.0, 1.0)))
    return np.array([data.qpos[0], data.qvel[0], theta, data.qvel[4]], dtype=np.float32)

def normalize_obs(obs):
    x,xd,th,thd = obs
    return np.array([
        np.clip(x  /X_NORM,  -1,1),
        np.clip(xd /XD_NORM, -1,1),
        np.clip(th /TH_NORM, -1,1),
        np.clip(thd/THD_NORM,-1,1),
    ], dtype=np.float32)


## Reward function

Same as Arthur.


In [3]:
def nav_reward(obs, prev_x, goal_x, at_goal, done, step):
    x,xd,theta,thd = obs
    if done:    return -20.0
    if at_goal: return  50.0
    r  = (abs(prev_x-goal_x) - abs(x-goal_x)) * 15.0
    r -= abs(x-goal_x) * 0.05
    r += 0.02
    if abs(theta)>0.20: r -= (abs(theta)-0.2)*5.0
    return float(r)


## SAC Networks

Identical to Arthur. `torque_max = 5.0 Nm`.


In [4]:
class NavActor(nn.Module):
    LOG_STD_MIN=-5; LOG_STD_MAX=2
    def __init__(self, torque_max=TORQUE_MAX):
        super().__init__(); self.torque_max=torque_max
        self.net=nn.Sequential(nn.Linear(5,256),nn.ReLU(),nn.Linear(256,256),nn.ReLU())
        self.mean_layer=nn.Linear(256,1); self.log_std_layer=nn.Linear(256,1)
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,0.5); nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.mean_layer.weight,0.01)
    def forward(self,x):
        h=self.net(x)
        return self.mean_layer(h), self.log_std_layer(h).clamp(self.LOG_STD_MIN,self.LOG_STD_MAX).exp()
    def get_action(self,obs,goal_x,deterministic=False):
        dist_norm=float(np.clip((goal_x-obs[0])/5.,-1,1))
        inp=torch.FloatTensor([*normalize_obs(obs),dist_norm]).unsqueeze(0)
        with torch.no_grad():
            mean,std=self(inp)
            if deterministic: raw=mean; logp=torch.zeros(1)
            else:
                d=torch.distributions.Normal(mean,std); raw=d.rsample()
                logp=(d.log_prob(raw)-torch.log(1-torch.tanh(raw).pow(2)+1e-6)).sum(-1)
            torque=torch.tanh(raw)*self.torque_max
        return torque.item(),logp,None
    def sample(self,obs_t):
        mean,std=self(obs_t); d=torch.distributions.Normal(mean,std); raw=d.rsample()
        logp=(d.log_prob(raw)-torch.log(1-torch.tanh(raw).pow(2)+1e-6)).sum(-1,keepdim=True)
        torque=torch.tanh(raw)*self.torque_max
        return torque,logp,torch.tanh(mean)*self.torque_max

class NavCritic(nn.Module):
    def __init__(self):
        super().__init__()
        mk=lambda: nn.Sequential(nn.Linear(6,256),nn.ReLU(),nn.Linear(256,256),nn.ReLU(),nn.Linear(256,1))
        self.q1,self.q2=mk(),mk()
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.orthogonal_(m.weight,0.5); nn.init.zeros_(m.bias)
    def forward(self,obs_t,act_t):
        x=torch.cat([obs_t,act_t],dim=-1); return self.q1(x),self.q2(x)


## Replay Buffer


In [7]:
class ReplayBuffer:
    def __init__(self,capacity=1_000_000,obs_dim=5):
        self.cap=capacity; self.ptr=self.size=0
        self.obs=np.zeros((capacity,obs_dim),dtype=np.float32)
        self.act=np.zeros((capacity,1),dtype=np.float32)
        self.rew=np.zeros((capacity,1),dtype=np.float32)
        self.obs2=np.zeros((capacity,obs_dim),dtype=np.float32)
        self.done=np.zeros((capacity,1),dtype=np.float32)
    def push(self,o,a,r,o2,d):
        self.obs[self.ptr]=o; self.act[self.ptr]=[a]
        self.rew[self.ptr]=r; self.obs2[self.ptr]=o2; self.done[self.ptr]=d
        self.ptr=(self.ptr+1)%self.cap; self.size=min(self.size+1,self.cap)
    def sample(self,n=256):
        i=np.random.randint(0,self.size,n)
        return (torch.FloatTensor(self.obs[i]),torch.FloatTensor(self.act[i]),
                torch.FloatTensor(self.rew[i]),torch.FloatTensor(self.obs2[i]),
                torch.FloatTensor(self.done[i]))
    def __len__(self): return self.size


## SAC `train_nav`

| Change | Reason |
|---|---|
| `ctrl[0]=-u`, no `ctrl[1]` | reaction wheel removed from model |
| `TARGET_ENT=0.0` | prevents alpha collapse (was -1.0) |
| `GROUND_Z=0.075` hardcoded | no lookup table needed |
| No scipy | fast inline quaternion math |
| Print every 10 ep, shows Avg10 + Avg100 | see progress faster |


In [8]:
def train_nav(x_start=0.0, x_goal=2.0, seed=42, tag="nav_sac_1d", max_steps=3_000_000):
    torch.manual_seed(seed); np.random.seed(seed)
    GAMMA,TAU,LR=0.99,0.005,3e-4
    BATCH,WARMUP,MAX_EP_STEPS=256,5000,3000
    TARGET_ENT=0.0   # KEY FIX: keeps alpha healthy throughout training

    actor=NavActor(torque_max=TORQUE_MAX)
    critic=NavCritic()
    critic_tgt=NavCritic(); critic_tgt.load_state_dict(critic.state_dict())
    for p in critic_tgt.parameters(): p.requires_grad=False
    actor_opt=optim.Adam(actor.parameters(),lr=LR)
    critic_opt=optim.Adam(critic.parameters(),lr=LR)
    log_alpha=torch.tensor(0.0,requires_grad=True)
    alpha_opt=optim.Adam([log_alpha],lr=LR)
    replay=ReplayBuffer(1_000_000,obs_dim=5)

    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)

    def reset_env():
        mujoco.mj_resetData(model,data)
        data.qpos[0]=x_start
        data.qpos[2]=GROUND_Z          # wheel radius, no lookup table
        data.qpos[3:7]=[1,0,0,0]       # upright
        data.qvel[:]=0.0
        data.qvel[4]=np.random.uniform(-0.01,0.01)  # tiny pitch-rate noise
        mujoco.mj_forward(model,data); return get_obs(data)

    def env_step(torque):
        u=float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX))
        data.ctrl[0]=-u             # same sign convention as Arthur
        mujoco.mj_step(model,data)  # only 1 actuator — no ctrl[1]
        obs=get_obs(data); x,_,th,_=obs
        done=abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE
        at_goal=abs(x-x_goal)<0.10 and abs(th)<0.25
        return obs,done,at_goal

    def make_inp(obs):
        return np.array([*normalize_obs(obs),float(np.clip((x_goal-obs[0])/5.,-1,1))],dtype=np.float32)

    def sac_update():
        if len(replay)<BATCH: return 0.0
        ob,ac,re,ob2,dn=replay.sample(BATCH)
        alpha=log_alpha.exp().detach()
        with torch.no_grad():
            na,nlp,_=actor.sample(ob2)
            qt=torch.min(*critic_tgt(ob2,na))-alpha*nlp
            y=re+GAMMA*(1-dn)*qt
        q1,q2=critic(ob,ac)
        cl=nn.MSELoss()(q1,y)+nn.MSELoss()(q2,y)
        critic_opt.zero_grad(); cl.backward()
        nn.utils.clip_grad_norm_(critic.parameters(),1.0); critic_opt.step()
        na,lp,_=actor.sample(ob)
        al=(alpha*lp-torch.min(*critic(ob,na))).mean()
        actor_opt.zero_grad(); al.backward()
        nn.utils.clip_grad_norm_(actor.parameters(),1.0); actor_opt.step()
        eal=-(log_alpha*(lp+TARGET_ENT).detach()).mean()
        alpha_opt.zero_grad(); eal.backward(); alpha_opt.step()
        with torch.no_grad():
            for p,pt in zip(critic.parameters(),critic_tgt.parameters()):
                pt.data.mul_(1-TAU); pt.data.add_(TAU*p.data)
        return cl.item()

    logger=RunLogger("SAC",seed,log_every_steps=2000)
    rewards=[]; losses=[]; best_avg=-9999; best_weights=None
    total_steps=ep_count=0; recent_rewards=[]

    hdr = "   Steps |    Ep |    Avg10 |   Avg100 |   Rec% |   Alpha |    CritL"
    print(f"\nSAC | A={x_start}m -> B={x_goal}m | budget={max_steps:,} steps")
    print(f"  TARGET_ENT={TARGET_ENT}  UPDATE_EVERY={UPDATE_EVERY}x{UPDATE_PER_STEP}")
    print(hdr); print("-"*len(hdr))

    while total_steps<max_steps:
        obs=reset_env(); ep_ret=0.0; prev_x=obs[0]; ep_step=0; ep_cls=[]; arrived_flag=False
        for _ in range(MAX_EP_STEPS):
            torque=(np.random.uniform(-TORQUE_MAX,TORQUE_MAX) if total_steps<WARMUP
                    else actor.get_action(obs,x_goal)[0])
            obs2,done,at_goal=env_step(torque)
            ep_step+=1; total_steps+=1
            r=nav_reward(obs2,prev_x,x_goal,at_goal,done,ep_step)
            ep_ret+=r; prev_x=obs2[0]
            replay.push(make_inp(obs),torque,r,make_inp(obs2),float(done or at_goal))
            if total_steps>=WARMUP and total_steps%UPDATE_EVERY==0:
                for _ in range(UPDATE_PER_STEP): ep_cls.append(sac_update())
            obs=obs2
            if done or at_goal or ep_step>=MAX_EP_STEPS: arrived_flag=at_goal; break
            if total_steps>=max_steps: break
        ep_count+=1
        ep_loss=float(np.mean(ep_cls)) if ep_cls else 0.0
        rewards.append(ep_ret); losses.append(ep_loss); recent_rewards.append(ep_ret)
        avg100,rp=logger.episode_end(total_steps,ep_count,ep_ret,arrived_flag,ep_loss)
        if ep_count%10==0:
            avg10=float(np.mean(recent_rewards[-10:]))
            al=float(log_alpha.exp().item())
            icon="OK" if avg100>0 else "UP" if avg100>-20 else ".."
            print(f"{total_steps:>8,} | {ep_count:>5} | {avg10:>8.2f} | {avg100:>8.2f} | "
                  f"{rp:>5.1f}% | {al:>7.4f} | {ep_loss:>8.3f}  {icon}",flush=True)
            if avg100>best_avg:
                best_avg=avg100
                best_weights={k:v.clone() for k,v in actor.state_dict().items()}
                torch.save(actor.state_dict(),f"{SAVE_DIR}/nav_sac_{tag}.pth")
    if best_weights: actor.load_state_dict(best_weights)
    return actor,rewards,losses,logger


In [9]:
def train_one_verify_many(algo_label, train_seed=42,
                          verify_seeds=(788,999,555,321,444),
                          x_start=0.0, x_goal=2.0, step_budget=500_000):
    verify_seeds=list(verify_seeds)
    print(f"\n{60*chr(61)}\n  {algo_label}  TRAIN seed={train_seed}\n{60*chr(61)}")
    net,rewards,losses,logger=train_nav(x_start=x_start,x_goal=x_goal,
        seed=train_seed,tag=f"cmp_seed{train_seed}",max_steps=step_budget)

    def eval_policy(net,seed,n=20):
        torch.manual_seed(seed); np.random.seed(seed)
        m=mujoco.MjModel.from_xml_path(XML_PATH); d=mujoco.MjData(m)
        strict=loose=fell=0; max_xs=[]; times=[]
        for _ in range(n):
            mujoco.mj_resetData(m,d)
            d.qpos[0]=x_start; d.qpos[2]=GROUND_Z
            d.qpos[3:7]=[1,0,0,0]; d.qvel[:]=0.0
            d.qvel[4]=np.random.uniform(-0.02,0.02)
            mujoco.mj_forward(m,d); obs=get_obs(d); mx=0.0; end="timeout"
            for step in range(3000):
                with torch.no_grad(): t,_,_=net.get_action(obs,x_goal,deterministic=True)
                d.ctrl[0]=-float(np.clip(t,-TORQUE_MAX,TORQUE_MAX))
                mujoco.mj_step(m,d); obs=get_obs(d); x,_,th,_=obs; mx=max(mx,x)
                if abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE: end="fell"; break
                if abs(x-x_goal)<0.10 and abs(th)<0.35: end="strict"; times.append(step*m.opt.timestep); break
                elif abs(x-x_goal)<0.25 and abs(th)<0.35: end="loose"; times.append(step*m.opt.timestep)
            strict+=end=="strict"; loose+=end=="loose"; fell+=end=="fell"; max_xs.append(mx)
        return {"strict_pct":strict/n*100,"loose_pct":(strict+loose)/n*100,
                "fell_pct":fell/n*100,"avg_max_x":float(np.mean(max_xs)),
                "avg_time":float(np.mean(times)) if times else 999.0}

    print(f"\n  Verifying on {len(verify_seeds)} seeds...")
    print(f"  Seed   | Strict% | Loose% | Fell% | Time"); print("  "+"-"*44)
    per_seed={}
    for vs in verify_seeds:
        r=eval_policy(net,vs); per_seed[vs]=r
        icon="OK" if r["strict_pct"]>=80 else "~" if r["strict_pct"]>=50 else "X"
        print(f"  {vs:>6} | {r['strict_pct']:>6.0f}% | {r['loose_pct']:>5.0f}% | "
              f"{r['fell_pct']:>4.0f}% | {r['avg_time']:>5.1f}s  {icon}")
    summary={"train_seed":train_seed,"verify_seeds":verify_seeds,
             "strict_pct":float(np.mean([per_seed[v]["strict_pct"] for v in verify_seeds])),
             "strict_std":float(np.std([per_seed[v]["strict_pct"] for v in verify_seeds])),
             "fell_pct":float(np.mean([per_seed[v]["fell_pct"] for v in verify_seeds])),
             "avg_time":float(np.mean([per_seed[v]["avg_time"] for v in verify_seeds
                                        if per_seed[v]["avg_time"]<999] or [999])),"per_seed":per_seed}
    print(f"\n  avg: {summary['strict_pct']:.1f}% +/- {summary['strict_std']:.1f}% strict | {summary['fell_pct']:.1f}% fell")
    logger.save(SAVE_DIR,f"cmp_seed{train_seed}",eval_results=summary)
    return net,logger,summary


## Recording, Plots


In [10]:
def record_all_seeds(nav_net,train_seeds=[42],verify_seeds=[788,999,555,321,444],
                     x_start=0.0,x_goal=2.0,max_steps=3000):
    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)
    dt=model.opt.timestep; renderer=mujoco.Renderer(model,height=480,width=640)
    cam=mujoco.MjvCamera(); cam.type=mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat=np.array([1.0,0.0,0.3]); cam.distance=4.5; cam.azimuth=90; cam.elevation=-15
    for seed,role in [(s,"TRAIN") for s in train_seeds]+[(s,"VERIFY") for s in verify_seeds]:
        torch.manual_seed(seed); np.random.seed(seed)
        mujoco.mj_resetData(model,data)
        data.qpos[0]=x_start; data.qpos[2]=GROUND_Z
        data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0; mujoco.mj_forward(model,data)
        obs=get_obs(data); frames=[]; strict=arrived=fell=False
        for step in range(max_steps):
            with torch.no_grad(): torque,_,_=nav_net.get_action(obs,x_goal,deterministic=True)
            data.ctrl[0]=-float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX))
            mujoco.mj_step(model,data); obs=get_obs(data); x,_,theta,_=obs
            fell=abs(x)>X_DONE_LIMIT or abs(theta)>TH_DONE
            arrived=abs(x-x_goal)<0.25 and abs(theta)<0.35
            strict=abs(x-x_goal)<0.10 and abs(theta)<0.35
            renderer.update_scene(data,camera=cam)
            img=PILImage.fromarray(renderer.render()); draw=ImageDraw.Draw(img); W,H=img.size
            prog=float(np.clip(x/x_goal,0,1)); bw=W-40
            pcol=(0,200,0) if strict else (255,140,0) if arrived else (30,100,220)
            draw.rectangle([20,8,W-20,28],fill=(40,40,40))
            draw.rectangle([20,8,20+int(bw*prog),28],fill=pcol)
            draw.text((22,10),"A",fill=(255,255,255)); draw.text((W-28,10),"B",fill=(255,255,255))
            draw.text((W//2-40,10),f"{prog*100:.0f}%  x={x:.3f}m",fill=(255,255,255))
            badge_col=(0,60,140) if role=="TRAIN" else (100,0,140)
            draw.rectangle([8,34,170,58],fill=badge_col)
            draw.text((12,38),f"[{role}] seed={seed}",fill=(255,255,255))
            if strict:    sc,st=(0,120,0),  f"STRICT x={x:.3f}m t={step*dt:.1f}s"
            elif arrived: sc,st=(120,100,0),f"LOOSE  x={x:.3f}m t={step*dt:.1f}s"
            elif fell:    sc,st=(140,0,0),  f"FELL   x={x:.3f}m th={np.degrees(theta):.1f}deg"
            else:         sc,st=(20,20,70), f"x={x:+.3f}m th={np.degrees(theta):+.1f}deg u={torque:+.2f}Nm t={step*dt:.1f}s"
            draw.rectangle([175,34,W-8,58],fill=sc); draw.text((178,38),st,fill=(255,255,255))
            frames.append(np.array(img))
            if fell or strict: break
        fname=f"{SAVE_DIR}/sac_{role.lower()}_seed{seed}.gif"
        imageio.mimsave(fname,frames,fps=30)
        status="STRICT" if strict else "LOOSE" if arrived else "FELL" if fell else "TIMEOUT"
        print(f"  [{role}] seed={seed} -> {status}  {fname}")


In [13]:
def plot_loss_reward(logger=None,rewards=None,losses=None,algo="model",save_prefix=None):
    save_prefix=save_prefix or algo.lower()
    if logger is not None:
        sx=[r["steps"] for r in logger.rows]; ry=[r["avg_reward"] for r in logger.rows]
        ly=[r["loss"] for r in logger.rows]; wt=getattr(logger,"total_wall_time",None); hs=True
    else:
        hs=False
    fig,(axr,axl)=plt.subplots(1,2,figsize=(15,5))
    title=f"{algo} Training"
    if hs and wt: title+=f"  (wall: {wt/60:.1f} min)"
    fig.suptitle(title,fontsize=13,fontweight="bold")
    if hs:
        axr.plot(sx,ry,color="#E74C3C",lw=2.5,label=f"avg100 reward (peak={max(ry):.1f})")
        axl.plot(sx,ly,color="#9B59B6",lw=2,label="Critic Loss")
        axr.set_xlabel("Steps"); axl.set_xlabel("Steps")
    else:
        r=np.array(rewards); w=min(100,max(2,len(r)//10))
        axr.plot(np.convolve(r,np.ones(w)/w,mode="valid"),color="#E74C3C",lw=2)
        l=np.array(losses); axl.plot(l,color="#9B59B6",alpha=0.5,lw=1)
        axr.set_xlabel("Episode"); axl.set_xlabel("Episode")
    axr.axhline(0,color="green",ls="--",alpha=0.5); axr.set_title("Reward",fontweight="bold"); axr.grid(alpha=0.3)
    axl.axhline(0,color="green",ls="--",alpha=0.5); axl.set_title("Critic Loss",fontweight="bold"); axl.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_training.png",dpi=150,bbox_inches="tight"); plt.show()


In [4]:
def plot_episode_traces(net,algo="model",x_goal=2.0,save_prefix=None):
    save_prefix=save_prefix or algo.lower()
    model=mujoco.MjModel.from_xml_path(XML_PATH); data=mujoco.MjData(model)
    dt=model.opt.timestep; mujoco.mj_resetData(model,data)
    data.qpos[0]=0.0; data.qpos[2]=GROUND_Z; data.qpos[3:7]=[1,0,0,0]; data.qvel[:]=0.0
    mujoco.mj_forward(model,data); obs=get_obs(data)
    ts,xs,ths,torqs=[],[],[],[]
    for step in range(3000):
        with torch.no_grad(): torque,_,_=net.get_action(obs,x_goal,deterministic=True)
        u=float(np.clip(torque,-TORQUE_MAX,TORQUE_MAX)); data.ctrl[0]=-u
        mujoco.mj_step(model,data); obs=get_obs(data); x,_,th,_=obs
        ts.append(step*dt); xs.append(x); ths.append(np.degrees(th)); torqs.append(torque)
        if abs(x)>X_DONE_LIMIT or abs(th)>TH_DONE or (abs(x-x_goal)<0.25 and abs(th)<0.35): break
    ts,xs,ths,torqs=np.array(ts),np.array(xs),np.array(ths),np.array(torqs)
    arrived=abs(xs[-1]-x_goal)<0.25 and abs(ths[-1])<20
    rms=np.sqrt(np.mean(torqs**2))
    fig=plt.figure(figsize=(15,9)); gs=gridspec.GridSpec(2,2,hspace=0.38,wspace=0.25)
    fig.suptitle(f"{algo} Episode",fontsize=13,fontweight="bold")
    ax=fig.add_subplot(gs[0,:]); ax.plot(ts,xs,"#E74C3C",lw=2.5,label="x")
    ax.axhline(0,color="blue",ls=":",lw=2); ax.axhline(x_goal,color="green",ls=":",lw=2,label=f"goal {x_goal}m")
    ax.axhspan(x_goal-0.25,x_goal+0.25,alpha=0.1,color="green")
    if arrived:
        idx=np.where(np.abs(xs-x_goal)<0.25)[0][0]
        ax.axvline(ts[idx],color="green",ls="--",alpha=0.6)
        ax.annotate(f"ARRIVED t={ts[idx]:.1f}s",xy=(ts[idx],xs[idx]),xytext=(ts[idx]+0.3,x_goal-0.4),
                    color="green",fontsize=9,arrowprops=dict(arrowstyle="->",color="green"))
    ax.set_xlabel("Time(s)"); ax.set_ylabel("x(m)"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax=fig.add_subplot(gs[1,0]); ax.plot(ts,ths,"#E74C3C",lw=2)
    ax.axhspan(-20,20,alpha=0.06,color="green"); ax.set_xlabel("Time(s)"); ax.set_ylabel("Tilt(deg)"); ax.grid(alpha=0.3)
    ax=fig.add_subplot(gs[1,1]); ax.plot(ts,torqs,"#2ECC71",lw=2,label=f"Torque RMS={rms:.2f}Nm")
    ax.axhline(TORQUE_MAX,color="orange",ls=":"); ax.axhline(-TORQUE_MAX,color="orange",ls=":")
    ax.set_xlabel("Time(s)"); ax.set_ylabel("Torque(Nm)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.savefig(f"{save_prefix}_traces.png",dpi=150,bbox_inches="tight"); plt.show()
    print(f"  Duration {ts[-1]:.1f}s  MaxX {max(xs):.3f}m  AvgTilt {np.mean(np.abs(ths)):.1f}deg  RMS {rms:.3f}Nm")


## RUN ALL


In [13]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim

print(sys.executable)
print(sys.version)
print(torch.__version__)

m = nn.Linear(1, 1)
opt = optim.Adam(m.parameters(), lr=1e-3)

print("PyTorch optimizer works.")

c:\Users\edward\anaconda3\python.exe
3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
2.12.0+cpu
PyTorch optimizer works.


In [5]:
import torch

net = NavActor(torque_max=TORQUE_MAX)

state = torch.load("SavedSeeds/nav_sac_cmp_seed42.pth", map_location="cpu")
net.load_state_dict(state)

net.eval()

print("Loaded saved SAC model successfully.")

Loaded saved SAC model successfully.


In [6]:
needed = ["torch", "np", "mujoco", "NavActor", "TORQUE_MAX", "XML_PATH", "get_obs"]

for name in needed:
    print(name, "OK" if name in globals() else "MISSING")

torch OK
np OK
mujoco OK
NavActor OK
TORQUE_MAX OK
XML_PATH OK
get_obs OK


In [7]:
obs = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

with torch.no_grad():
    torque, _, _ = net.get_action(obs, 2.0, deterministic=True)

print("torque =", torque)

torque = -3.593120813369751


In [8]:
model = mujoco.MjModel.from_xml_path(XML_PATH)
data = mujoco.MjData(model)

mujoco.mj_resetData(model, data)
data.qpos[0] = 0.0
data.qpos[2] = GROUND_Z
data.qpos[3:7] = [1, 0, 0, 0]
data.qvel[:] = 0.0
mujoco.mj_forward(model, data)

for i in range(1000):
    data.ctrl[0] = 0.0
    mujoco.mj_step(model, data)

print("MuJoCo step test worked.")

MuJoCo step test worked.


In [9]:
model = mujoco.MjModel.from_xml_path(XML_PATH)
data = mujoco.MjData(model)

mujoco.mj_resetData(model, data)
data.qpos[0] = 0.0
data.qpos[2] = GROUND_Z
data.qpos[3:7] = [1, 0, 0, 0]
data.qvel[:] = 0.0
mujoco.mj_forward(model, data)

obs = get_obs(data)

for step in range(1000):
    with torch.no_grad():
        torque, _, _ = net.get_action(obs, 2.0, deterministic=True)

    data.ctrl[0] = -float(np.clip(torque, -TORQUE_MAX, TORQUE_MAX))
    mujoco.mj_step(model, data)
    obs = get_obs(data)

print("SAC + MuJoCo worked.")
print("final obs =", obs)

SAC + MuJoCo worked.
final obs = [ 1.8034595   1.8572817  -0.07026283 -1.0696261 ]


In [10]:
import pandas as pd
import numpy as np
import torch
import mujoco

def save_episode_trace_csv(net, filename="sac_episode_trace.csv", x_goal=2.0):
    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data = mujoco.MjData(model)

    mujoco.mj_resetData(model, data)
    data.qpos[0] = 0.0
    data.qpos[2] = GROUND_Z
    data.qpos[3:7] = [1, 0, 0, 0]
    data.qvel[:] = 0.0
    mujoco.mj_forward(model, data)

    obs = get_obs(data)
    dt = model.opt.timestep

    rows = []

    for step in range(3000):
        with torch.no_grad():
            torque, _, _ = net.get_action(obs, x_goal, deterministic=True)

        torque = float(torque)
        u = float(np.clip(torque, -TORQUE_MAX, TORQUE_MAX))

        data.ctrl[0] = -u
        mujoco.mj_step(model, data)

        obs = get_obs(data)
        x, xdot, theta, thetadot = obs

        rows.append({
            "time": step * dt,
            "x": float(x),
            "xdot": float(xdot),
            "theta_rad": float(theta),
            "theta_deg": float(np.degrees(theta)),
            "thetadot": float(thetadot),
            "torque": torque,
        })

        fell = abs(x) > X_DONE_LIMIT or abs(theta) > TH_DONE
        arrived = abs(x - x_goal) < 0.25 and abs(theta) < 0.35

        if fell or arrived:
            break

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)

    print("Saved:", filename)
    print("Final x:", df["x"].iloc[-1])
    print("Final theta deg:", df["theta_deg"].iloc[-1])
    print("Final time:", df["time"].iloc[-1])

save_episode_trace_csv(net)

Saved: sac_episode_trace.csv
Final x: 1.7504280805587769
Final theta deg: -1.949341893196106
Final time: 1.97


In [21]:
record_all_seeds(net)

  [TRAIN] seed=42 -> STRICT  SavedSeeds/sac_train_seed42.gif


KeyboardInterrupt: 

In [52]:
net, logger, summary = train_one_verify_many("SAC", train_seed=42)



  SAC  TRAIN seed=42


AttributeError: partially initialized module 'torch._dynamo' from 'c:\Users\edward\anaconda3\Lib\site-packages\torch\_dynamo\__init__.py' has no attribute 'decorators' (most likely due to a circular import)

In [ ]:
record_all_seeds(net, train_seeds=[42], verify_seeds=[788,999,555], x_goal=2.0)


In [11]:
plot_loss_reward(logger=logger, algo="SAC")


NameError: name 'plot_loss_reward' is not defined

In [22]:
plot_episode_traces(net, algo="SAC")


: 

In [14]:
import json
with open("SavedSeeds/SAC_cmp_seed42.json") as f:
    log_data = json.load(f)

# Rebuild a logger-like object from the file
class FakeLogger:
    def __init__(self, data):
        self.rows = data["rows"]
        self.total_wall_time = data.get("total_wall_time", 0)

logger = FakeLogger(log_data)
plot_loss_reward(logger=logger, algo="SAC")

: 